In [1]:
import sys

# Copy and paste file path.
path = r''
sys.path.append(path)
import StochasticLanguages as sl
import numpy as np

## Example 1

In [2]:
# Comparing svd and canonical methods of obtaining GLM

alphabet = list('ab')
pi = np.array([0.5, 0.5])
T = {'a': np.array([[2/3, 0],
                    [0, 1/3]]),
     'b': np.array([[0, 1/3],
                    [2/3, 0]])}
ones = np.ones(2)
G = sl.MPS(T, pi, ones)
seq_fun = G.get_seq_fun(9)
sl.disp_seq_fun(G.get_seq_fun(3))

L = sl.Language(alphabet, seq_fun)
W1, left1, right1 = L.get_quasi_realisation(4, type='svd')
W2, left2, right2 = L.get_quasi_realisation(4, type='canonical')

M1 = sl.MPS(W1, left1, right1)
M2 = sl.MPS(W2, left2, right2)
sl.disp_seq_fun(M1.get_seq_fun(3))
sl.disp_seq_fun(M2.get_seq_fun(3))

print(np.allclose(list(M1.get_seq_fun(5).values()), list(M2.get_seq_fun(5).values()), atol=1e-10))

Sequence Function:
*: 1
a: 0.5
b: 0.5
aa: 0.27778
ab: 0.22222
ba: 0.27778
bb: 0.22222
aaa: 0.16667
aab: 0.11111
aba: 0.11111
abb: 0.11111
baa: 0.16667
bab: 0.11111
bba: 0.11111
bbb: 0.11111
Obtaining quasi-realisation...
Quasi-realisation obtained.
Obtaining quasi-realisation...
Quasi-realisation obtained.
Sequence Function:
*: 1
a: 0.5
b: 0.5
aa: 0.27778
ab: 0.22222
ba: 0.27778
bb: 0.22222
aaa: 0.16667
aab: 0.11111
aba: 0.11111
abb: 0.11111
baa: 0.16667
bab: 0.11111
bba: 0.11111
bbb: 0.11111
Sequence Function:
*: 1
a: 0.5
b: 0.5
aa: 0.27778
ab: 0.22222
ba: 0.27778
bb: 0.22222
aaa: 0.16667
aab: 0.11111
aba: 0.11111
abb: 0.11111
baa: 0.16667
bab: 0.11111
bba: 0.11111
bbb: 0.11111
True


In [4]:
# Verify that HMM realisation of M1 gives the correct sequence function.

T1, v1, o1 = sl.find_HMM_realisation(W1, left1, right1)

print(T1, v1, o1)

G1 = sl.MPS(T1, v1, o1)
sl.disp_seq_fun(G1.get_seq_fun(3))

Finding HMM realisation...
HMM of rank 2 found. Loss = 2.5251351737116916e-25
{'a': array([[ 3.33333333e-01, -2.29308689e-13],
       [ 1.40321504e-13,  6.66666667e-01]]), 'b': array([[ 5.11570050e-14,  6.66666667e-01],
       [ 3.33333333e-01, -5.11345626e-14]])} [0.5 0.5] [1. 1.]
Sequence Function:
*: 1
a: 0.5
b: 0.5
aa: 0.27778
ab: 0.22222
ba: 0.27778
bb: 0.22222
aaa: 0.16667
aab: 0.11111
aba: 0.11111
abb: 0.11111
baa: 0.16667
bab: 0.11111
bba: 0.11111
bbb: 0.11111


In [5]:
# Verify that HMM realisation of M2 gives the correct sequence function.

T2, v2, o2 = sl.find_HMM_realisation(W2, left2, right2)

print(T2, v2, o2)

G2 = sl.MPS(T2, v2, o2)
sl.disp_seq_fun(G2.get_seq_fun(3))


Finding HMM realisation...
HMM of rank 2 found. Loss = 4.5551151766777e-24
{'a': array([[ 6.66666667e-01,  5.18474153e-13],
       [-5.56036698e-13,  3.33333333e-01]]), 'b': array([[-4.76877797e-13,  3.33333333e-01],
       [ 6.66666667e-01,  4.78691161e-13]])} [0.5 0.5] [1. 1.]
Sequence Function:
*: 1
a: 0.5
b: 0.5
aa: 0.27778
ab: 0.22222
ba: 0.27778
bb: 0.22222
aaa: 0.16667
aab: 0.11111
aba: 0.11111
abb: 0.11111
baa: 0.16667
bab: 0.11111
bba: 0.11111
bbb: 0.11111


## Example 2

In [3]:
# Random QHMM to GLM
# Note: Somehow this randomly generated QHMM has a minimal HMM realisation, which is very surprising as this does not usually happen. Most randomly generated QHMMs have no minimal HMM realisation.

alphabet = list('ab')
dim = 2
all_kraus = sl.gen_random_kraus_operators(dim, 0, 1, 2, 3) # parameters are seeds for Kraus operators
transition_matrices = {'a': sl.get_transfer_matrix(all_kraus[:2]), 'b': sl.get_transfer_matrix(all_kraus[2:])}
left_vector = sl.gen_random_density_matrix(dim, 0).reshape(-1, order='F').conj().T
right_vector = np.eye(dim).reshape(-1, order='F')

Q = sl.MPS(transition_matrices, left_vector, right_vector)

sl.disp_seq_fun(Q.get_seq_fun(4))

L = sl.Language(alphabet, Q.get_seq_fun(12))
print('Rank = ', np.linalg.matrix_rank(L.get_hankel(5, 5)))
W, left, right = L.get_quasi_realisation(5, type='svd')

T, lvec, rvec = sl.find_HMM_realisation(W, left, right)
print(T, lvec, rvec)
if T is not None:
    H = sl.MPS(T, lvec, rvec)
    sl.disp_seq_fun(H.get_seq_fun(4))

Sequence Function:
*: 1
a: 0.2035
b: 0.7965
aa: 0.039931
ab: 0.16357
ba: 0.20925
bb: 0.58725
aaa: 0.00734
aab: 0.032591
aba: 0.042974
abb: 0.1206
baa: 0.040621
bab: 0.16862
bba: 0.14673
bbb: 0.44052
aaaa: 0.001375
aaab: 0.005965
aaba: 0.0085917
aabb: 0.023999
abaa: 0.008633
abab: 0.034341
abba: 0.030047
abbb: 0.090552
baaa: 0.0076896
baab: 0.032931
baba: 0.044181
babb: 0.12444
bbaa: 0.028094
bbab: 0.11863
bbba: 0.11124
bbbb: 0.32928
Rank =  4
Obtaining quasi-realisation...
Quasi-realisation obtained.
Finding HMM realisation...


C:\Users\Cheel\OneDrive - Nanyang Technological University\Courses\FYP\Codes\StochasticLanguages.py:24: ComplexWarning: Casting complex values to real discards the imaginary part
  hankel[i,j] = self.seq_fun.get(prefix + suffix, 0.0) # returns 0.0 if prefix + suffix is not in seq_fun


HMM of rank 4 found. Loss = 1.393941871428816e-28
{'a': array([[1.28101648e-03, 7.60983248e-04, 2.30317207e-01, 5.88201942e-02],
       [1.75763357e-01, 9.38727525e-02, 4.14422950e-02, 1.13553845e-01],
       [2.85888091e-07, 1.84945508e-02, 2.76230969e-02, 3.80457379e-02],
       [6.15876131e-02, 2.27731786e-06, 1.00370111e-01, 5.01238848e-02]]), 'b': array([[2.87388602e-01, 1.03976702e-01, 1.22731093e-01, 1.94724201e-01],
       [2.98494963e-01, 2.01917203e-03, 2.40368947e-01, 3.44846678e-02],
       [5.22027476e-02, 4.53126631e-01, 4.07640940e-01, 2.86601026e-03],
       [6.03366095e-01, 3.42926132e-02, 4.17589108e-08, 1.50257364e-01]])} [0.15402954 0.13190932 0.38149016 0.33257097] [1. 1. 1. 1.]
Sequence Function:
*: 1
a: 0.2035
b: 0.7965
aa: 0.039931
ab: 0.16357
ba: 0.20925
bb: 0.58725
aaa: 0.00734
aab: 0.032591
aba: 0.042974
abb: 0.1206
baa: 0.040621
bab: 0.16862
bba: 0.14673
bbb: 0.44052
aaaa: 0.001375
aaab: 0.005965
aaba: 0.0085917
aabb: 0.023999
abaa: 0.008633
abab: 0.034341
a

## Function Tests

In [2]:
# Testing reshuffle function

arrK = sl.gen_random_kraus_operators(2, 20, 13, 50, 17, 15, 17, 19, 1235)
arrK = arrK[1:6]
transfer = sl.get_transfer_matrix(arrK)
choi = sl.get_choi_matrix(arrK)
print(np.allclose(transfer, sl.reshuffle(choi), atol=1e-10))
print(np.allclose(choi, sl.reshuffle(transfer), atol=1e-10))

True
True
